In [ ]:
!pip install pyvi

In [ ]:

import torch

data_path = "IWSLT'15 en-vi/"
train_data_path = '/kaggle/input/iwslt15-englishvietnamese/IWSLT\'15 en-vi/'
saved_model_path = '/kaggle/working/'
saved_tokenizer_path = '/kaggle/working/'
test_data_path = 'data/test_data/'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Data parameters
MAX_SEQ_LEN = 60
ENGLISH_VOCAB_SIZE = 32000  # NEW: Limited vocab size
VIETNAMESE_VOCAB_SIZE = 32000  # NEW: Limited vocab size

# Model architecture
NUM_LAYERS = 6
D_MODEL = 512
D_FF = 2048
EPS = 0.1
BATCH_SIZE = 128
NUM_HEADS = 8
EPOCHS = 25
DROPOUT = 0.2
CLIP = 1.0
BATCH_PRINT = 100



UNKNOWN_TOKEN = '<unk>'
PAD_TOKEN = '<pad>'
START_TOKEN = '<start>'
END_TOKEN = '<end>'
PAD_TOKEN_POS = 0

# Tokenizing

500K dataset: Use 32K shared vocab with BPE/SentencePiece


1M dataset: Use 40K-50K shared vocab with BPE/SentencePiece

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from pyvi.ViTokenizer import ViTokenizer
from keras.src.legacy.preprocessing.text import Tokenizer
from keras.src.utils import pad_sequences

class TranslationDataset(Dataset):
    def __init__(self, en_sequences, vi_sequences, max_length):
        self.en_sequences = en_sequences
        self.vi_sequences = vi_sequences
        self.max_length = max_length
        
    def __len__(self):
        return len(self.en_sequences)
    
    def __getitem__(self, idx):
        en_seq = self.en_sequences[idx]
        vi_seq = self.vi_sequences[idx]
        
        if len(en_seq) < self.max_length:
            en_seq = en_seq + [PAD_TOKEN_POS] * (self.max_length - len(en_seq))
        if len(vi_seq) < self.max_length:
            vi_seq = vi_seq + [PAD_TOKEN_POS] * (self.max_length - len(vi_seq))
            
        return torch.tensor(en_seq, dtype=torch.long), torch.tensor(vi_seq, dtype=torch.long)


In [ ]:
def load_data(en_file, vi_file):
    with open(en_file, 'r', encoding='utf-8') as f:
        en_data = f.read().strip().split("\n")
    with open(vi_file, 'r', encoding='utf-8') as f:
        vi_data = f.read().strip().split("\n")
    return en_data, vi_data

def get_tokenize(data, add_start_end=False, max_vocab_size=None):
    tokenizer = Tokenizer(
        filters='', 
        oov_token=UNKNOWN_TOKEN
    )
    
    if add_start_end:
        tokenizer.fit_on_texts([START_TOKEN, END_TOKEN] + data)
    else:
        tokenizer.fit_on_texts(data)
    
    if max_vocab_size is not None:
        sorted_words = sorted(tokenizer.word_counts.items(), key=lambda x: x[1], reverse=True)
        top_words = dict(sorted_words[:max_vocab_size])
        
        new_word_index = {UNKNOWN_TOKEN: 1}
        if add_start_end:
            new_word_index[START_TOKEN] = len(new_word_index) + 1
            new_word_index[END_TOKEN] = len(new_word_index) + 1
        
        idx = len(new_word_index) + 1
        for word in top_words:
            if word not in new_word_index:
                new_word_index[word] = idx
                idx += 1
        
        tokenizer.word_index = new_word_index
        tokenizer.index_word = {v: k for k, v in new_word_index.items()}
        
        print(f"   Vocabulary reduced to {len(new_word_index):,} tokens")
    
    return data, tokenizer

def get_tokenize_seq(en_data, vi_data, en_tokenizer, vi_tokenizer, max_sequence_length):

    en_sequences = en_tokenizer.texts_to_sequences(en_data)
    
    vi_data = [ViTokenizer.tokenize(sentence) for sentence in vi_data]
    vi_data_with_tokens = [f"{START_TOKEN} {sentence} {END_TOKEN}" for sentence in vi_data]
    vi_sequences = vi_tokenizer.texts_to_sequences(vi_data_with_tokens)
    
    filtered_en = []
    filtered_vi = []
    for i in range(len(en_sequences)):
        if (len(en_sequences[i]) <= max_sequence_length) and (len(vi_sequences[i]) <= max_sequence_length):
            filtered_en.append(en_sequences[i])
            filtered_vi.append(vi_sequences[i])
    
    return filtered_en, filtered_vi

def preprocess_tokenizer(en_data, vi_data):
    print("Creating English tokenizer (source - no START/END, limit: 32,000)...")
    en_data, en_tokenizer = get_tokenize(
        en_data, 
        add_start_end=False, 
        max_vocab_size=32000
    )
    
    print("Creating Vietnamese tokenizer (target - with START/END, limit: 32,000)...")
    vi_data = [ViTokenizer.tokenize(sentence) for sentence in vi_data]
    vi_data, vi_tokenizer = get_tokenize(
        vi_data,
        add_start_end=True,  
        max_vocab_size=32000
    )
    
    return en_tokenizer, vi_tokenizer

def preprocess_data(train_src_path, train_trg_path, val_src_path, val_trg_path):
    en_data, vi_data = load_data(train_src_path, train_trg_path)
    en_data_val, vi_data_val = load_data(val_src_path, val_trg_path)
    
    en_tokenizer, vi_tokenizer = preprocess_tokenizer(en_data, vi_data)
    
    en_sequences, vi_sequences = get_tokenize_seq(
        en_data, vi_data, en_tokenizer, vi_tokenizer, max_sequence_length=MAX_SEQ_LEN
    )
    en_val_sequences, vi_val_sequences = get_tokenize_seq(
        en_data_val, vi_data_val, en_tokenizer, vi_tokenizer, max_sequence_length=MAX_SEQ_LEN
    )
    
    all_train_sequences = TranslationDataset(en_sequences, vi_sequences, MAX_SEQ_LEN)
    all_val_sequences = TranslationDataset(en_val_sequences, vi_val_sequences, MAX_SEQ_LEN)
    
    return en_tokenizer, vi_tokenizer, all_train_sequences, all_val_sequences

# Multi Head Attention

In [ ]:
from torch import nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()

        assert d_model% num_heads==0
        
        self.num_heads = num_heads
        
        self.attention = ScaleDotProductAttention()
        
        self.w_q = nn.Linear(d_model, d_model)
        
        self.w_k = nn.Linear(d_model, d_model)
        
        self.w_v = nn.Linear(d_model, d_model)
        
        self.w_concat = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        query, key, value = self.w_q(query), self.w_k(key), self.w_v(value)

        query, key, value = self.split(query), self.split(key), self.split(value)

        out, attention = self.attention(query, key, value, mask=mask)

        out = self.concat(out)
        out = self.w_concat(out)

        return out

    def split(self, tensor):
        batch_size, length, d_model = tensor.size()

        d_tensor = d_model // self.num_heads
        tensor = tensor.view(batch_size, length, self.num_heads, d_tensor).transpose(1, 2)

        return tensor

    def concat(self, tensor):
        batch_size, num_heads, length, d_tensor = tensor.size()
        d_model = d_tensor * self.num_heads

        tensor = tensor.transpose(1, 2).contiguous().view(batch_size, length, d_model)
        return tensor

# Learning Rate Schedule

In [ ]:
from torch.optim.lr_scheduler import _LRScheduler

class LearningRateSchedule(_LRScheduler):
    def __init__(self, optimizer, initial_lr, decay_rates, decay_steps, lr_decay_interval, last_epoch=-1):
        """
        initial_lr: Learning rate ban đầu
        decay_rates: Danh sách hệ số decay (n phần tử)
        decay_steps: Danh sách step ứng với decay (n-1 phần tử)
        lr_decay_interval: Khoảng cách giữa các lần decay
        """
        assert len(decay_rates) - 1 == len(decay_steps), "Số lượng decay_steps phải ít hơn decay_rates một phần tử"

        self.initial_lr = initial_lr
        self.decay_rates = decay_rates
        self.decay_steps = decay_steps
        self.lr_decay_interval = lr_decay_interval
        self.prev_decay_step = 0

        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        step = self.last_epoch
        lr = self.initial_lr
        prev_decay_step = 0

   
        for i in range(len(self.decay_steps)):
            decay_factor = self.decay_rates[i]
            num_intervals = max((min(step, self.decay_steps[i]) - prev_decay_step) // self.lr_decay_interval, 0)
            lr *= decay_factor ** num_intervals
            prev_decay_step = self.decay_steps[i]

   
        decay_factor = self.decay_rates[-1]
        num_intervals = max((step - prev_decay_step) // self.lr_decay_interval, 0)
        lr *= decay_factor ** num_intervals

        return [lr for _ in self.base_lrs]  

    def state_dict(self):
        return {
            "initial_lr": self.initial_lr,
            "decay_rates": self.decay_rates,
            "decay_steps": self.decay_steps,
            "lr_decay_interval": self.lr_decay_interval,
            "prev_decay_step": self.prev_decay_step
        }

    def load_state_dict(self, state_dict):
        self.initial_lr = state_dict["initial_lr"]
        self.decay_rates = state_dict["decay_rates"]
        self.decay_steps = state_dict["decay_steps"]
        self.lr_decay_interval = state_dict["lr_decay_interval"]
        self.prev_decay_step = state_dict["prev_decay_step"]

In [ ]:
class TransformerLRSchedule:
    """
    Custom learning rate scheduler for Transformer models.
    Implements warmup followed by decay, with optional max_lr cap.
    """
    def __init__(self, optimizer, d_model, warmup_steps, factor=1.0, max_lr=None):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.factor = factor
        self.max_lr = max_lr  # Optional maximum learning rate cap
        self.current_step = 0
        
    def step(self):
        """Update learning rate based on current step"""
        self.current_step += 1
        lr = self.get_lr()
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
    
    def get_lr(self):
        """Calculate learning rate for current step"""
        step = max(self.current_step, 1)  
        
        lr = self.factor * (self.d_model ** -0.5) * min(
            step ** -0.5, 
            step * (self.warmup_steps ** -1.5)
        )
        
        if self.max_lr is not None:
            lr = min(lr, self.max_lr)
        
        return lr
    
    def get_last_lr(self):
        return [self.get_lr()]


WARMUP_STEPS = 1000 
FACTOR = 1.0  


# Scaled Dot Product

In [ ]:
import torch
from torch import nn
import math

class ScaleDotProductAttention(nn.Module):
    def __init__(self):
        super(ScaleDotProductAttention, self).__init__()
        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, query, key, value, mask=None):
        batch_size, num_heads, length, d_tensor = key.size()
        
        key_t = key.transpose(2, 3)
        score = (query @ key_t) / math.sqrt(d_tensor)
        
        if mask is not None:
            score = score.masked_fill(mask == 0, -1e4)
        
        score = self.softmax(score)
        value = score @ value
        
        return value, score


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, device):

        super(PositionalEncoding, self).__init__()

        self.encoding = torch.zeros(max_len, d_model, device=device)
        self.encoding.requires_grad = False 

        pos = torch.arange(0, max_len, device=device)
        pos = pos.float().unsqueeze(dim=1)

        _2i = torch.arange(0, d_model, 2, device=device).float()

        self.encoding[:, 0::2] = torch.sin(pos / (10000 ** (_2i / d_model)))
        self.encoding[:, 1::2] = torch.cos(pos / (10000 ** (_2i / d_model)))

    def forward(self, x):
        batch_size, seq_len = x.size()
        return self.encoding[:seq_len, :]

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super(PositionwiseFeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

class TransformerEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, dropout, device):
        super(TransformerEmbedding, self).__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_TOKEN_POS)
        self.pos_emb = PositionalEncoding(d_model, max_len, device)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        tok_emb = self.tok_emb(x)
        pos_emb = self.pos_emb(x)
        return self.dropout(tok_emb + pos_emb)


# Encoder

In [ ]:
from torch import nn

class EncoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, num_heads, dropout):
        super(EncoderLayer, self).__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model, eps=EPS)
        self.dropout1 = nn.Dropout(dropout)

        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model, eps=EPS)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        _x = x
        x = self.attention(x, x, x, src_mask)

        x = self.dropout1(x)
        x = self.norm1(_x + x)

        _x = x
        x = self.ffn(x)

        x = self.dropout2(x)
        x = self.norm2(_x + x)

        return x

class Encoder(nn.Module):
    def __init__(self, inp_vocab_size, max_len, d_model, d_ff, num_heads, num_layers, dropout, device):
        super(Encoder, self).__init__()
        self.emb = TransformerEmbedding(inp_vocab_size, d_model, max_len, dropout, device=device)
        self.layers = nn.ModuleList([EncoderLayer(d_model, d_ff, num_heads, dropout) for _ in range(num_layers)])

    def forward(self, src, src_mask):
        x = self.emb(src)
        for layer in self.layers:
            x = layer(x, src_mask)

        return x

# Decoder

In [ ]:
from torch import nn

class Decoder_Layer(nn.Module):
    def __init__(self, d_model, d_ff, num_heads, dropout):
        super(Decoder_Layer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model, eps=EPS)
        self.dropout1 = nn.Dropout(dropout)

        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model, eps=EPS)
        self.dropout2 = nn.Dropout(dropout)

        self.ffn = PositionwiseFeedForward(d_model, d_ff, DROPOUT)
        self.norm3 = nn.LayerNorm(d_model, eps=EPS)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_out, trg_mask, src_mask):
        _x = x
        x = self.self_attn(x, x, x, mask=trg_mask)

        x = self.dropout1(x)
        x = self.norm1(_x + x)

        if enc_out is not None:
            _x = x
            x = self.enc_dec_attn(x, enc_out, enc_out, mask=src_mask)

            x = self.dropout2(x)
            x = self.norm2(_x + x)

        _x = x
        x = self.ffn(x)

        x = self.dropout3(x)
        x = self.norm3(_x + x)

        return x

class Decoder(nn.Module):
    def __init__(self, trg_vocab_size, max_len, d_model, d_ff, num_heads, num_layers, dropout, device):
        super(Decoder, self).__init__()
        self.embedding = TransformerEmbedding(trg_vocab_size, d_model, max_len, dropout, device)
        self.layers = nn.ModuleList([Decoder_Layer(d_model, d_ff, num_heads, dropout) for i in range(num_layers)])
        self.linear = nn.Linear(d_model, trg_vocab_size)

    def forward(self, trg, enc_src, trg_mask, src_mask):
        trg = self.embedding(trg)

        for layer in self.layers:
            trg = layer(trg, enc_src, trg_mask, src_mask)

        output = self.linear(trg)

        return output


# Transformer

In [ ]:
import torch
from torch import nn

class Transformer(nn.Module):
    def __init__(self, src_pad_idx, trg_pad_idx, inp_vocab_size, trg_vocab_size, d_model, num_heads, max_len, d_ff, num_layers, dropout, device):
        super(Transformer, self).__init__()
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.device = device

        self.encoder = Encoder(inp_vocab_size, max_len, d_model, d_ff, num_heads, num_layers, dropout, device)
        self.decoder = Decoder(trg_vocab_size, max_len, d_model, d_ff, num_heads, num_layers, dropout, device)

    def forward(self, src, trg):
        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)
        enc_out = self.encoder(src, src_mask)
        output = self.decoder(trg, enc_out, trg_mask, src_mask)
        return output

    def make_src_mask(self, src):
        src_mask = (src != self.src_pad_idx).unsqueeze(dim=1).unsqueeze(dim=2)
        return src_mask

    def make_trg_mask(self, trg):
        trg_pad_mask = (trg != self.trg_pad_idx).unsqueeze(dim=1).unsqueeze(dim=3)
        trg_len = trg.shape[1]
        trg_look_ahead_mask = torch.tril(torch.ones(trg_len, trg_len)).bool().to(self.device)
        trg_mask = trg_pad_mask & trg_look_ahead_mask

        return trg_mask

# Training

In [ ]:
import torch
import math
import time
import gc
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from pyvi.ViTokenizer import ViTokenizer
from keras.src.legacy.preprocessing.text import Tokenizer

# ==================== CONFIGURATION ====================
RESUME_TRAINING = True  # Set to True to continue training
CHECKPOINT_PATH = '/kaggle/input/trained-models/model-en-vi-2.352-0.531_32k_32k_4_epoches.pt'
TRAINED_EPOCHS = 4  # How many epochs already completed

# ==================== TRAINING FUNCTIONS ====================

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

def print_gpu_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f'GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved')

def train(model, iterator, optimizer, criterion, clip, scaler, scheduler, accumulation_steps=2):
    model.train()
    epoch_loss = 0
    total_correct = 0
    total_tokens = 0
    optimizer.zero_grad()
    
    for i, (src, trg) in enumerate(iterator):
        src = src.to(model.device, non_blocking=True)
        trg = trg.to(model.device, non_blocking=True)
        
        # src is now English, trg is Vietnamese
        with torch.amp.autocast('cuda'):
            output = model(src, trg[:, :-1])
            output_reshape = output.contiguous().view(-1, output.shape[-1])
            trg_reshaped = trg[:, 1:].contiguous().view(-1)
            loss = criterion(output_reshape, trg_reshaped)
            loss = loss / accumulation_steps
        
        scaler.scale(loss).backward()
        
        if (i + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            
            if (i + 1) % (accumulation_steps * 10) == 0:
                torch.cuda.empty_cache()
        
        with torch.no_grad():
            pred = output.argmax(dim=-1).view(-1)
            mask = (trg_reshaped != PAD_TOKEN_POS)
            correct = (pred == trg_reshaped) & mask
            total_correct += correct.sum().item()
            total_tokens += mask.sum().item()
        
        epoch_loss += loss.item() * accumulation_steps
        
        if (i + 1) % BATCH_PRINT == 0:
            lr = optimizer.param_groups[0]['lr']
            acc = total_correct / total_tokens if total_tokens > 0 else 0
            print(f'Batch: {i+1}/{len(iterator)}, Loss: {loss.item() * accumulation_steps:.4f}, '
                  f'Accuracy: {acc:.4f}, LR: {lr:.6f}')
        
        del src, trg, output, output_reshape, trg_reshaped, loss, pred, mask, correct
            
    return epoch_loss / len(iterator), total_correct / total_tokens

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    total_correct = 0
    total_tokens = 0
    
    with torch.no_grad():
        for i, (src, trg) in enumerate(iterator):
            src = src.to(model.device, non_blocking=True)
            trg = trg.to(model.device, non_blocking=True)
            
            with torch.amp.autocast('cuda'):
                output = model(src, trg[:, :-1])
                output_reshape = output.contiguous().view(-1, output.shape[-1])
                trg_reshaped = trg[:, 1:].contiguous().view(-1)
                loss = criterion(output_reshape, trg_reshaped)

            pred = output.argmax(dim=-1).view(-1)
            mask = (trg_reshaped != PAD_TOKEN_POS)
            correct = (pred == trg_reshaped) & mask
            total_correct += correct.sum().item()
            total_tokens += mask.sum().item()
            epoch_loss += loss.item()
            
            del src, trg, output, output_reshape, trg_reshaped, loss, pred, mask, correct
            
            if (i + 1) % 50 == 0:
                torch.cuda.empty_cache()

    return epoch_loss / len(iterator), total_correct / total_tokens

def run(total_epoch, best_loss, start_epoch=0, accumulation_steps=2):
    train_losses, test_losses = [], []
    
    for step in range(start_epoch, start_epoch + total_epoch):
        print(f'\n{"="*60}')
        print(f'Epoch: {step + 1}/{start_epoch + total_epoch}')
        print(f'{"="*60}')
        
        start_time = time.time()
        
        try:
            train_loss, train_accuracy = train(
                model, train_batches, optimizer, criterion, 
                CLIP, scaler, scheduler, accumulation_steps
            )
            
            torch.cuda.empty_cache()
            gc.collect()
            
            val_loss, val_accuracy = evaluate(model, val_batches, criterion)
            
            end_time = time.time()
            epoch_mins, epoch_secs = epoch_time(start_time, end_time)
            
            train_losses.append(train_loss)
            test_losses.append(val_loss)
            
            if val_loss < best_loss:
                best_loss = val_loss
                torch.save(model.state_dict(), 
                          f'{saved_model_path}/model-en-vi-{val_loss:.3f}-{val_accuracy:.3f}.pt')
                print(f'✓ Model saved! Best val loss: {best_loss:.3f}')

            print(f'\nEpoch {step + 1} Summary:')
            print(f'  Time: {epoch_mins}m {epoch_secs}s')
            print(f'  Train Loss: {train_loss:.3f} | Train Acc: {train_accuracy:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
            print(f'  Val Loss: {val_loss:.3f} | Val Acc: {val_accuracy:.3f} | Val PPL: {math.exp(val_loss):7.3f}')
            
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f'\n!!! OOM Error at epoch {step + 1} !!!')
                print('Try reducing: batch size, MAX_SEQ_LEN, or model size')
                print_gpu_memory()
                torch.cuda.empty_cache()
                gc.collect()
                raise e
            else:
                raise e
        
        torch.cuda.empty_cache()
        gc.collect()
    
    return train_losses, test_losses



In [ ]:
# # ==================== MAIN EXECUTION ====================

# # Clear GPU cache
# torch.cuda.empty_cache()
# gc.collect()

# print("="*60)
# print("ENGLISH -> VIETNAMESE TRANSLATION MODEL")
# print("="*60)
# print("\n" + "="*60)
# print("PREPROCESSING DATA")
# print("="*60)

# # Load and preprocess data
# en_tokenizer, vi_tokenizer, all_train_sequences, all_val_sequences = preprocess_data(
#     "/kaggle/input/train-en-vi/train_2456580.en",
#     "/kaggle/input/train-en-vi/train_2456580.vi",
#     data_path + "tst2013.en.txt", 
#     data_path + "tst2013.vi.txt"
# )

# REDUCED_BATCH_SIZE = 128

# # ==================== SAVE TOKENIZERS ====================
# print("\nSaving tokenizers...")
# import pickle

# try:
#     with open(f'{saved_tokenizer_path}/en_tokenizer_src.pkl', 'wb') as f:
#         pickle.dump(en_tokenizer, f)
        
#     with open(f'{saved_tokenizer_path}/vi_tokenizer_trg.pkl', 'wb') as f:
#         pickle.dump(vi_tokenizer, f)
        
#     print("✓ Tokenizers saved to:", saved_tokenizer_path)
# except Exception as e:
#     print(f"⚠ Warning: Could not save tokenizers: {e}")

# # Create DataLoaders
# train_batches = DataLoader(
#     all_train_sequences, 
#     batch_size=REDUCED_BATCH_SIZE, 
#     shuffle=True,
#     pin_memory=False,
#     num_workers=0
# )
# val_batches = DataLoader(
#     all_val_sequences, 
#     batch_size=REDUCED_BATCH_SIZE, 
#     shuffle=False,
#     pin_memory=False,
#     num_workers=0
# )

# # Get vocab sizes (SWAPPED for EN->VI)
# en_vocab_size = len(en_tokenizer.word_index) + 1  # Source
# vi_vocab_size = len(vi_tokenizer.word_index) + 1  # Target

# print(f"\nVocabulary sizes:")
# print(f"  English (source): {en_vocab_size:,}")
# print(f"  Vietnamese (target): {vi_vocab_size:,}")
# print(f"\nDataset info:")
# print(f"  Train samples: {len(all_train_sequences):,}")
# print(f"  Val samples: {len(all_val_sequences):,}")
# print(f"  Batches per epoch: {len(train_batches):,}")
# print(f"  Batch size: {REDUCED_BATCH_SIZE}")

# print("\n" + "="*60)
# print("INITIALIZING MODEL")
# print("="*60)

# # Initialize model (SWAPPED vocab sizes)
# model = Transformer(
#     src_pad_idx=PAD_TOKEN_POS,
#     trg_pad_idx=PAD_TOKEN_POS,
#     d_model=D_MODEL,
#     inp_vocab_size=en_vocab_size,  # English is input
#     trg_vocab_size=vi_vocab_size,  # Vietnamese is target
#     max_len=MAX_SEQ_LEN,
#     d_ff=D_FF,
#     num_heads=NUM_HEADS,
#     num_layers=NUM_LAYERS,
#     dropout=DROPOUT,
#     device=DEVICE
# ).to(DEVICE)

# # ==================== LOAD CHECKPOINT IF RESUMING ====================
# if RESUME_TRAINING:
#     print("\n" + "="*60)
#     print("LOADING PRETRAINED MODEL")
#     print("="*60)
    
#     model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
#     print(f"✓ Successfully loaded model from: {CHECKPOINT_PATH}")
    
#     import re
#     match = re.search(r'model-en-vi-(\d+\.\d+)-', CHECKPOINT_PATH)
#     if match:
#         BEST_LOSS = float(match.group(1))
#         print(f"✓ Previous best validation loss: {BEST_LOSS:.3f}")
#     else:
#         BEST_LOSS = float('inf')
#         print("⚠ Could not extract loss from filename")
# else:
#     BEST_LOSS = float('inf')

# print(f'\nModel has {count_parameters(model):,} trainable parameters')
# print_gpu_memory()

# # ==================== OPTIMIZER ====================
# optimizer = optim.AdamW(
#     model.parameters(),
#     lr=0,
#     betas=(0.9, 0.98),
#     eps=1e-9,
#     weight_decay=1e-4
# )

# # ==================== SCHEDULER ====================
# if RESUME_TRAINING:
#     WARMUP_STEPS_USE = WARMUP_STEPS
#     MAX_LR_USE = 3e-4
    
#     scheduler = TransformerLRSchedule(
#         optimizer=optimizer,
#         d_model=D_MODEL,  
#         warmup_steps=WARMUP_STEPS_USE,
#         factor=FACTOR,
#         max_lr=MAX_LR_USE
#     )
    
#     batches_per_epoch = len(train_batches) // 2
#     scheduler.current_step = TRAINED_EPOCHS * batches_per_epoch
#     print(f"\n✓ Scheduler resuming from step {scheduler.current_step:,}")
#     print(f"✓ Current learning rate: {scheduler.get_lr():.6f}")
# else:
#     scheduler = TransformerLRSchedule(
#         optimizer=optimizer,
#         d_model=D_MODEL,  
#         warmup_steps=WARMUP_STEPS,
#         factor=FACTOR,
#         max_lr=3e-4
#     )

# # ==================== LOSS FUNCTION ====================
# criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN_POS)

# # ==================== MIXED PRECISION SCALER ====================
# scaler = torch.amp.GradScaler('cuda')

# print("\n" + "="*60)
# print("TRAINING CONFIGURATION")
# print("="*60)
# print(f"  Translation direction: English → Vietnamese")
# print(f"  Actual batch size: {REDUCED_BATCH_SIZE}")
# print(f"  Gradient accumulation steps: 2")
# print(f"  Effective batch size: {REDUCED_BATCH_SIZE * 2}")
# print(f"  Weight updates per epoch: {len(train_batches) // 2}")
# print(f"  Total epochs: {EPOCHS}")

# print("\n" + "="*60)
# print("STARTING TRAINING")
# print("="*60)

# # Calculate remaining epochs
# if RESUME_TRAINING:
#     REMAINING_EPOCHS = EPOCHS - TRAINED_EPOCHS
#     print(f"\n⚠ Resuming from epoch {TRAINED_EPOCHS}")
#     print(f"⚠ Training {REMAINING_EPOCHS} additional epochs")
#     print(f"⚠ Total epochs will be: {EPOCHS}")
    
#     train_losses, test_losses = run(
#         total_epoch=REMAINING_EPOCHS,
#         best_loss=BEST_LOSS,
#         start_epoch=TRAINED_EPOCHS,
#         accumulation_steps=2
#     )
# else:
#     train_losses, test_losses = run(
#         total_epoch=EPOCHS, 
#         best_loss=BEST_LOSS,
#         start_epoch=0,
#         accumulation_steps=2
#     )

# print("\n" + "="*60)
# print("TRAINING COMPLETE!")
# print("="*60)
# print(f"Final best validation loss: {min(test_losses):.3f}")
# print(f"Model saved with prefix: model-en-vi-*.pt")

In [ ]:
!pip install sacrebleu

# Evaluate Without beam seach

In [ ]:
# # ==================== STANDALONE BLEU EVALUATION SECTION ====================
# # Run this AFTER training is complete
# # ENGLISH -> VIETNAMESE TRANSLATION EVALUATION

# import torch
# from sacrebleu.metrics import BLEU

# print("\n" + "="*60)
# print("ENGLISH -> VIETNAMESE BLEU EVALUATION")
# print("="*60)

# # ==================== CONFIGURATION ====================
# EVAL_CHECKPOINT_PATH = '/kaggle/input/trained-models/model-en-vi-2.283-0.538_32k_32k_6_epoches.pt'  # Your trained EN->VI model
# EVAL_BATCH_SIZE = 32  # Smaller batch for translation
# EVAL_MAX_LEN = 60  # Max translation length

# # ==================== TRANSLATION FUNCTIONS ====================

# def translate_sentence(model, src_tensor, en_tokenizer, vi_tokenizer, device, max_len=60):
#     """
#     Translate a single English sentence to Vietnamese using greedy decoding
    
#     Args:
#         model: Trained EN->VI model
#         src_tensor: English source sentence tensor
#         en_tokenizer: English tokenizer (source)
#         vi_tokenizer: Vietnamese tokenizer (target)
#         device: torch device
#         max_len: Maximum translation length
    
#     Returns:
#         translation: Vietnamese translation string
#     """
#     model.eval()
    
#     with torch.no_grad():
#         # Add batch dimension if needed
#         if src_tensor.dim() == 1:
#             src_tensor = src_tensor.unsqueeze(0)
#         src_tensor = src_tensor.to(device)
        
#         # Start with START token (Vietnamese)
#         trg_indices = [vi_tokenizer.word_index[START_TOKEN]]
        
#         for i in range(max_len):
#             trg_tensor = torch.LongTensor(trg_indices).unsqueeze(0).to(device)
            
#             with torch.amp.autocast('cuda'):
#                 output = model(src_tensor, trg_tensor)
            
#             # Get the predicted next token
#             pred_token = output.argmax(2)[:, -1].item()
#             trg_indices.append(pred_token)
            
#             # Stop if END token is predicted
#             if pred_token == vi_tokenizer.word_index[END_TOKEN]:
#                 break
        
#         # Convert indices to words (Vietnamese)
#         trg_tokens = [vi_tokenizer.index_word.get(idx, UNKNOWN_TOKEN) 
#                       for idx in trg_indices]
        
#         # Remove START and END tokens
#         trg_tokens = [token for token in trg_tokens 
#                       if token not in [START_TOKEN, END_TOKEN]]
        
#         return ' '.join(trg_tokens)

# def evaluate_bleu_corpus(model, data_loader, en_tokenizer, vi_tokenizer, device, 
#                          max_samples=None, verbose=True):
#     """
#     Calculate BLEU score on entire corpus (English -> Vietnamese)
    
#     Args:
#         model: Trained EN->VI translation model
#         data_loader: DataLoader with validation data (EN source, VI target)
#         en_tokenizer: English tokenizer (source)
#         vi_tokenizer: Vietnamese tokenizer (target)
#         device: torch device
#         max_samples: Maximum number of samples to evaluate (None = all)
#         verbose: Print progress
    
#     Returns:
#         bleu_score: SacreBLEU score (0-100)
#         hypotheses: List of model translations (Vietnamese)
#         references: List of reference translations (Vietnamese)
#     """
#     model.eval()
    
#     hypotheses = []  # Model predictions (Vietnamese)
#     references = []  # Ground truth translations (Vietnamese)
    
#     total_samples = 0
    
#     if verbose:
#         print(f"\nTranslating validation set (EN -> VI)...")
#         print(f"Device: {device}")
    
#     with torch.no_grad():
#         for batch_idx, (src, trg) in enumerate(data_loader):
#             batch_size = src.size(0)
            
#             for j in range(batch_size):
#                 if max_samples and total_samples >= max_samples:
#                     break
                
#                 # Get source sentence (English)
#                 src_sentence = src[j]
                
#                 # Translate English -> Vietnamese
#                 translation = translate_sentence(
#                     model, src_sentence, en_tokenizer, vi_tokenizer, device, max_len=EVAL_MAX_LEN
#                 )
#                 hypotheses.append(translation)
                
#                 # Get reference translation (Vietnamese)
#                 trg_indices = trg[j].cpu().tolist()
#                 trg_tokens = []
#                 for idx in trg_indices:
#                     if idx == PAD_TOKEN_POS:
#                         continue
#                     if idx == vi_tokenizer.word_index.get(START_TOKEN, -1):
#                         continue
#                     if idx == vi_tokenizer.word_index.get(END_TOKEN, -1):
#                         break
#                     word = vi_tokenizer.index_word.get(idx, '')
#                     if word:
#                         trg_tokens.append(word)
                
#                 reference = ' '.join(trg_tokens)
#                 references.append([reference])  # SacreBLEU expects list of lists
                
#                 total_samples += 1
            
#             if max_samples and total_samples >= max_samples:
#                 break
            
#             if verbose and (batch_idx + 1) % 20 == 0:
#                 print(f"  Processed {total_samples} sentences...")
    
#     if verbose:
#         print(f"  Total sentences translated: {total_samples}")
    
#     # Calculate BLEU score
#     bleu = BLEU()
#     bleu_score = bleu.corpus_score(hypotheses, references)
    
#     return bleu_score.score, hypotheses, references

# def print_translation_examples(hypotheses, references, num_examples=10):
#     """Print example translations (EN->VI)"""
#     print(f"\n{'='*60}")
#     print("TRANSLATION EXAMPLES (English -> Vietnamese)")
#     print(f"{'='*60}")
    
#     for i in range(min(num_examples, len(hypotheses))):
#         print(f"\nExample {i+1}:")
#         print(f"  Reference (VI): {references[i][0]}")
#         print(f"  Predicted (VI): {hypotheses[i]}")
#         print(f"  {'-'*58}")

# # ==================== LOAD VALIDATION DATA ====================

# print("\nLoading validation data...")

# # OPTION 1: If you saved tokenizers during training (recommended)
# # en_tokenizer = joblib.load('/kaggle/working/en_tokenizer_src.pkl')
# # vi_tokenizer = joblib.load('/kaggle/working/vi_tokenizer_trg.pkl')

# # OPTION 2: Quick recreation (faster than full preprocessing)
# print("Creating tokenizers (this may take a moment)...")
# en_data_train, vi_data_train = load_data(
#     "/kaggle/input/train-en-vi/train_2456580.en",
#     "/kaggle/input/train-en-vi/train_2456580.vi"
# )
# en_tokenizer, vi_tokenizer = preprocess_tokenizer(en_data_train, vi_data_train)

# # Now load ONLY validation data (don't reprocess training data)
# print("Loading validation sequences...")
# en_data_val, vi_data_val = load_data(
#     data_path + "tst2013.en.txt", 
#     data_path + "tst2013.vi.txt"
# )

# en_val_sequences, vi_val_sequences = get_tokenize_seq(
#     en_data_val, vi_data_val, en_tokenizer, vi_tokenizer, 
#     max_sequence_length=MAX_SEQ_LEN
# )

# # Dataset returns (English, Vietnamese) for EN->VI
# all_val_sequences = TranslationDataset(en_val_sequences, vi_val_sequences, MAX_SEQ_LEN)

# eval_loader = DataLoader(
#     all_val_sequences, 
#     batch_size=EVAL_BATCH_SIZE, 
#     shuffle=False,
#     pin_memory=False,
#     num_workers=0
# )

# print(f"✓ Validation samples: {len(all_val_sequences):,}")

# # ==================== LOAD TRAINED MODEL ====================

# print("\n" + "="*60)
# print("LOADING TRAINED EN->VI MODEL FOR EVALUATION")
# print("="*60)

# # Get vocab sizes (SWAPPED for EN->VI)
# en_vocab_size = len(en_tokenizer.word_index) + 1  # Source
# vi_vocab_size = len(vi_tokenizer.word_index) + 1  # Target

# # Initialize model with same architecture (EN->VI)
# eval_model = Transformer(
#     src_pad_idx=PAD_TOKEN_POS,
#     trg_pad_idx=PAD_TOKEN_POS,
#     d_model=D_MODEL,
#     inp_vocab_size=en_vocab_size,  # English input
#     trg_vocab_size=vi_vocab_size,  # Vietnamese output
#     max_len=MAX_SEQ_LEN,
#     d_ff=D_FF,
#     num_heads=NUM_HEADS,
#     num_layers=NUM_LAYERS,
#     dropout=DROPOUT,
#     device=DEVICE
# ).to(DEVICE)

# # Load checkpoint
# eval_model.load_state_dict(torch.load(EVAL_CHECKPOINT_PATH, map_location=DEVICE))
# eval_model.eval()

# print(f"✓ Model loaded from: {EVAL_CHECKPOINT_PATH}")
# print(f"✓ Model has {count_parameters(eval_model):,} parameters")
# print(f"✓ Translation direction: English -> Vietnamese")

# # ==================== RUN EVALUATION ====================

# print("\n" + "="*60)
# print("CALCULATING BLEU SCORE")
# print("="*60)

# # Full evaluation on entire validation set
# bleu_score, hypotheses, references = evaluate_bleu_corpus(
#     model=eval_model,
#     data_loader=eval_loader,
#     en_tokenizer=en_tokenizer,
#     vi_tokenizer=vi_tokenizer,
#     device=DEVICE,
#     max_samples=None,  # Evaluate all samples (set to 500 for quick test)
#     verbose=True
# )

# # ==================== DISPLAY RESULTS ====================

# print("\n" + "="*60)
# print("EVALUATION RESULTS (EN->VI)")
# print("="*60)
# print(f"\n✓ BLEU Score: {bleu_score:.2f}")
# print(f"✓ Total sentences evaluated: {len(hypotheses):,}")

# # Show translation examples
# print_translation_examples(hypotheses, references, num_examples=15)

# # ==================== ADDITIONAL METRICS (OPTIONAL) ====================

# print("\n" + "="*60)
# print("ADDITIONAL STATISTICS")
# print("="*60)

# # Calculate average translation length
# avg_hyp_len = sum(len(h.split()) for h in hypotheses) / len(hypotheses)
# avg_ref_len = sum(len(r[0].split()) for r in references) / len(references)

# print(f"Average hypothesis length: {avg_hyp_len:.1f} words")
# print(f"Average reference length: {avg_ref_len:.1f} words")
# print(f"Length ratio: {avg_hyp_len/avg_ref_len:.2f}")

# # Count empty translations
# empty_translations = sum(1 for h in hypotheses if len(h.strip()) == 0)
# print(f"Empty translations: {empty_translations} ({empty_translations/len(hypotheses)*100:.1f}%)")

# print("\n" + "="*60)
# print("EVALUATION COMPLETE")
# print("="*60)

# # ==================== INTERACTIVE TRANSLATION (BONUS) ====================

# def translate_custom_sentence(sentence, model, en_tokenizer, vi_tokenizer, device):
#     """
#     Translate a custom English sentence to Vietnamese
    
#     Args:
#         sentence: English sentence (string)
#         model: Trained EN->VI model
#         en_tokenizer: English tokenizer (source)
#         vi_tokenizer: Vietnamese tokenizer (target)
#         device: torch device
    
#     Returns:
#         translation: Vietnamese translation (string)
#     """
#     # Tokenize English (no special preprocessing needed)
#     sequence = en_tokenizer.texts_to_sequences([sentence])[0]
    
#     # Pad to max length
#     if len(sequence) < MAX_SEQ_LEN:
#         sequence = sequence + [PAD_TOKEN_POS] * (MAX_SEQ_LEN - len(sequence))
#     else:
#         sequence = sequence[:MAX_SEQ_LEN]
    
#     # Convert to tensor
#     src_tensor = torch.tensor(sequence, dtype=torch.long)
    
#     # Translate English -> Vietnamese
#     translation = translate_sentence(model, src_tensor, en_tokenizer, vi_tokenizer, device)
    
#     return translation

# # Example: Translate custom sentences
# print("\n" + "="*60)
# print("CUSTOM TRANSLATION EXAMPLES (EN -> VI)")
# print("="*60)

# custom_sentences = [
#     "I love learning machine learning.",
#     "The weather is nice today.",
#     "We are working on a machine translation project."
# ]

# for sent in custom_sentences:
#     translation = translate_custom_sentence(sent, eval_model, en_tokenizer, vi_tokenizer, DEVICE)
#     print(f"\nEnglish: {sent}")
#     print(f"Vietnamese: {translation}")

# print("\n" + "="*60)
# print("Note: Vietnamese output may use underscore-separated tokens")
# print("(e.g., 'máy_học' instead of 'máy học') due to PyVi tokenization")
# print("="*60)

In [ ]:
!pip install sacrebleu

# Evaluate With beam search

In [2]:
# ==================== SIMPLE EN -> VI EVALUATION (BEAM SEARCH + PROGRESS BAR) ====================

import torch
import joblib
import numpy as np
from sacrebleu.metrics import BLEU, CHRF
from torch.utils.data import DataLoader
from tqdm import tqdm   # ✅ IMPORTANT: Kaggle-safe tqdm

# ==================== CONFIG ====================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EVAL_CHECKPOINT_PATH = "models/model-en-vi-2.283-0.538_32k_32k_6_epoches.pt"
TOKENIZER_PATH = "tokenizers/"

EVAL_BATCH_SIZE = 32
MAX_LEN = 60
BEAM_SIZE = 4

# ==================== LOAD TOKENIZERS ====================

en_tokenizer = joblib.load(TOKENIZER_PATH + "en_vi_tokenizer_src_32k.pkl")
vi_tokenizer = joblib.load(TOKENIZER_PATH + "en_vi_vi_tokenizer_trg_32k.pkl")

print(f"English vocab: {len(en_tokenizer.word_index)+1}")
print(f"Vietnamese vocab: {len(vi_tokenizer.word_index)+1}")

START_ID = vi_tokenizer.word_index[START_TOKEN]
END_ID = vi_tokenizer.word_index[END_TOKEN]

# ==================== BEAM SEARCH DECODER ====================

def beam_search_decode(model, src_tensor, max_len=60, beam_size=4):
    model.eval()

    src_tensor = src_tensor.unsqueeze(0).to(DEVICE)
    beams = [([START_ID], 0.0)]  # (sequence, log_prob)

    with torch.no_grad():
        for _ in range(max_len):
            new_beams = []

            for seq, score in beams:
                if seq[-1] == END_ID:
                    new_beams.append((seq, score))
                    continue

                trg_tensor = torch.LongTensor(seq).unsqueeze(0).to(DEVICE)
                output = model(src_tensor, trg_tensor)
                log_probs = torch.log_softmax(output[:, -1], dim=-1)

                topk = torch.topk(log_probs, beam_size)

                for i in range(beam_size):
                    token = topk.indices[0][i].item()
                    token_score = topk.values[0][i].item()
                    new_beams.append((seq + [token], score + token_score))

            beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]

        best_seq = beams[0][0]

    tokens = [
        vi_tokenizer.index_word.get(i, "<unk>")
        for i in best_seq
        if i not in (START_ID, END_ID)
    ]

    return " ".join(tokens)

# ==================== EVALUATION LOOP (WITH PROGRESS BAR) ====================

def evaluate_en_vi(model, dataloader):
    model.eval()

    hypotheses = []
    references = []

    total_sentences = len(dataloader.dataset)
    print("\nStarting EN → VI beam search evaluation...")
    pbar = tqdm(total=total_sentences, unit="sent", ncols=100)

    with torch.no_grad():
        for src, trg in dataloader:
            src = src.to(DEVICE)
            trg = trg.to(DEVICE)

            batch_size = src.size(0)

            for i in range(batch_size):
                # --- Translate ---
                pred = beam_search_decode(
                    model,
                    src[i],
                    max_len=MAX_LEN,
                    beam_size=BEAM_SIZE
                )
                hypotheses.append(pred)

                # --- Reference ---
                ref_tokens = []
                for idx in trg[i].tolist():
                    if idx == PAD_TOKEN_POS:
                        continue
                    if idx == START_ID:
                        continue
                    if idx == END_ID:
                        break
                    w = vi_tokenizer.index_word.get(idx, "")
                    if w:
                        ref_tokens.append(w)

                references.append([" ".join(ref_tokens)])
                pbar.update(1)

    pbar.close()
    return hypotheses, references

# ==================== LOAD VALIDATION DATA ====================

print("\nLoading validation data...")

en_val, vi_val = load_data(
    data_path + "tst2013.en.txt",
    data_path + "tst2013.vi.txt"
)

en_seq, vi_seq = get_tokenize_seq(
    en_val,
    vi_val,
    en_tokenizer,
    vi_tokenizer,
    MAX_SEQ_LEN
)

val_dataset = TranslationDataset(en_seq, vi_seq, MAX_SEQ_LEN)

val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"Validation samples: {len(val_dataset)}")

# ==================== LOAD MODEL ====================

print("\nLoading model...")

model = Transformer(
    src_pad_idx=PAD_TOKEN_POS,
    trg_pad_idx=PAD_TOKEN_POS,
    d_model=D_MODEL,
    inp_vocab_size=len(en_tokenizer.word_index)+1,
    trg_vocab_size=len(vi_tokenizer.word_index)+1,
    max_len=MAX_SEQ_LEN,
    d_ff=D_FF,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    device=DEVICE
).to(DEVICE)

model.load_state_dict(torch.load(EVAL_CHECKPOINT_PATH, map_location=DEVICE))
model.eval()

print("Model loaded.")

# ==================== RUN EVAL ====================

hyps, refs = evaluate_en_vi(model, val_loader)

# ==================== METRICS ====================

bleu = BLEU()
chrf = CHRF(word_order=2)

bleu_score = bleu.corpus_score(hyps, refs)
chrf_score = chrf.corpus_score(hyps, refs)

# ==================== RESULTS ====================

print("\n" + "="*60)
print("EN → VI RESULTS (TOKENIZED, PyVi, BEAM SEARCH)")
print("="*60)
print(f"BLEU : {bleu_score.score:.2f}")
print(f"chrF : {chrf_score.score:.2f}")
print("="*60)

# ==================== SAMPLE OUTPUTS ====================

print("\nTranslation examples:\n")

for i in range(min(10, len(hyps))):
    print("-"*80)
    print("REF:", refs[i][0])
    print("HYP:", hyps[i])


English vocab: 32002
Vietnamese vocab: 32004


NameError: name 'START_TOKEN' is not defined